# Phase 3: Fine-Tuning with LoRA and Prompt Tuning (Using S3)
In this notebook, we will experiment with **LoRA (testing different target layers)** and **Prompt Tuning**. Data is loaded directly from S3.

In [ ]:
#!pip install -q mlflow boto3 transformers torch pandas scikit-learn datasets peft tqdm accelerate python-dotenv s3fs

### 1. Load Credentials & Set Up MLflow

In [1]:
import os
import mlflow
from dotenv import load_dotenv

# Load environment variables from .env if present
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME") or "finance-sentiment-mlflow-artifacts-kavishka"
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI") or "http://13.235.68.121:5000/"

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION

# Connect to your remote MLflow server
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("Connected to MLflow at:", mlflow.get_tracking_uri())
print("Using S3 bucket:", S3_BUCKET_NAME)
mlflow.set_experiment("Sentiment_FineTuning_Benchmark")

Connected to MLflow at: http://13.235.68.121:5000/
Using S3 bucket: finance-sentiment-mlflow-artifacts-kavishka


<Experiment: artifact_location='s3://finance-sentiment-mlflow-artifacts-kavishka/1', creation_time=1788882302611, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788882302611, lifecycle_stage='active', name='Sentiment_FineTuning_Benchmark', tags={}, trace_location=None, workspace='default'>

### 2. Read Datasets Directly From S3 & Prepare

In [2]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

print(f"Fetching datasets from S3 Bucket: {S3_BUCKET_NAME} ...")
train_df = pd.read_csv(f"s3://{S3_BUCKET_NAME}/train.csv")
val_df = pd.read_csv(f"s3://{S3_BUCKET_NAME}/val.csv")
test_df = pd.read_csv(f"s3://{S3_BUCKET_NAME}/test.csv")
print("✅ Datasets loaded successfully!")

label2id = {'positive': 0, 'negative': 1, 'neutral': 2}
id2label = {0: 'positive', 1: 'negative', 2: 'neutral'}

for df in [train_df, val_df, test_df]:
    df['label'] = df['Sentiment'].map(label2id)

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['Headline', 'label']]),
    'val': Dataset.from_pandas(val_df[['Headline', 'label']]),
    'test': Dataset.from_pandas(test_df[['Headline', 'label']])
})

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['Headline'], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

print("Data tokenization complete!")

Fetching datasets from S3 Bucket: finance-sentiment-mlflow-artifacts-kavishka ...
✅ Datasets loaded successfully!


Map:   0%|          | 0/2186 [00:00<?, ? examples/s]

Map:   0%|          | 0/263 [00:00<?, ? examples/s]

Map:   0%|          | 0/265 [00:00<?, ? examples/s]

Data tokenization complete!


### Experiment 1: LoRA (Targeting Attention Layers Only)

In [4]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import time

In [7]:
import time
import mlflow
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Initialize Base Model
model_lora_attn = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3, 
    id2label=id2label, 
    label2id=label2id
)

# 2. Configure LoRA for Attention Layers
lora_config_attn = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)
model_lora_attn = get_peft_model(model_lora_attn, lora_config_attn)
model_lora_attn.print_trainable_parameters()

# 3. TrainingArguments configured to graph training & validation loss curves in MLflow
training_args_attn = TrainingArguments(
    output_dir="./results_lora_attn",
    eval_strategy="epoch",          # Calculates & logs eval_loss at each epoch end
    logging_strategy="steps",       # Logs training loss during epochs
    logging_steps=20,               # Records training loss every 20 batches (draws a smooth curve)
    report_to="mlflow",             # Streams all loss & metric points straight into MLflow
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)

# 4. Initialize Trainer
trainer_lora_attn = Trainer(
    model=model_lora_attn,
    args=training_args_attn,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    compute_metrics=compute_metrics,
)

# 5. Execute Training & Logging
with mlflow.start_run(run_name="LoRA_Attention_Only"):
    start_time = time.time()
    
    # Automatically logs training loss and eval loss curves to MLflow
    trainer_lora_attn.train()
    
    train_time = time.time() - start_time
    mlflow.log_metric("train_time_seconds", train_time)
    
    # Evaluate on held-out test set
    test_results = trainer_lora_attn.evaluate(
        tokenized_datasets['test'], 
        metric_key_prefix="test"
    )
    
    print("\n--- Test Set Results ---")
    print(test_results)
    
    # Log final test metrics
    mlflow.log_metric("test_accuracy", test_results['test_accuracy'])
    mlflow.log_metric("test_f1_macro", test_results['test_f1_macro'])
    mlflow.log_metric("test_loss", test_results['test_loss'])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.107010,1.101949,0.326996,0.241379
2,1.105055,1.100208,0.346008,0.301370
3,1.110945,1.100093,0.346008,0.303508


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
1.110945,1.102838,3,0.328302,0.294092



--- Test Set Results ---
{'test_loss': 1.1028378009796143, 'test_accuracy': 0.3283018867924528, 'test_f1_macro': 0.29409183771406633}
🏃 View run LoRA_Attention_Only at: http://13.235.68.121:5000/#/experiments/1/runs/82cd9f91c96e45cb83b9bb55075cb2e0
🧪 View experiment at: http://13.235.68.121:5000/#/experiments/1


### Experiment 2: LoRA (Targeting All Linear Layers)

In [8]:
model_lora_all = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, id2label=id2label, label2id=label2id)

lora_config_all = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "dense"]
)
model_lora_all = get_peft_model(model_lora_all, lora_config_all)
model_lora_all.print_trainable_parameters()

training_args_all = TrainingArguments(
    output_dir="./results_lora_all_linear",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="mlflow"
)

trainer_lora_all = Trainer(
    model=model_lora_all,
    args=training_args_all,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    compute_metrics=compute_metrics,
)

with mlflow.start_run(run_name="LoRA_All_Linear"):
    start_time = time.time()
    trainer_lora_all.train()
    train_time = time.time() - start_time
    mlflow.log_metric("train_time_seconds", train_time)
    
    test_results = trainer_lora_all.evaluate(tokenized_datasets['test'])
    mlflow.log_metric("test_accuracy", test_results['eval_accuracy'])
    mlflow.log_metric("test_f1_macro", test_results['eval_f1_macro'])
    print("Test Results:", test_results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,341,699 || all params: 110,826,246 || trainable%: 1.2106


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.102896,0.330798,0.254148
2,No log,1.100989,0.330798,0.291945
3,No log,1.100736,0.315589,0.288378


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
No log,1.104574,3,0.313208,0.292867


Test Results: {'eval_loss': 1.1045743227005005, 'eval_accuracy': 0.3132075471698113, 'eval_f1_macro': 0.292866553736119}
🏃 View run LoRA_All_Linear at: http://13.235.68.121:5000/#/experiments/1/runs/fa92ff6c70824c709075ad71b8ca4d22
🧪 View experiment at: http://13.235.68.121:5000/#/experiments/1


### Experiment 3: Prompt Tuning

In [6]:
from peft import PromptTuningConfig, PromptTuningInit

model_prompt = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, id2label=id2label, label2id=label2id)

prompt_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    prompt_tuning_init=PromptTuningInit.RANDOM, # No text used!
    num_virtual_tokens=8
)

model_prompt = get_peft_model(model_prompt, prompt_config)
model_prompt.print_trainable_parameters()

training_args_prompt = TrainingArguments(
    output_dir="./results_prompt_tuning",
    eval_strategy="epoch",
    learning_rate=3e-2,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none"
)

trainer_prompt = Trainer(
    model=model_prompt,
    args=training_args_prompt,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    compute_metrics=compute_metrics,
)

with mlflow.start_run(run_name="Prompt_Tuning"):
    start_time = time.time()
    trainer_prompt.train()
    train_time = time.time() - start_time
    mlflow.log_metric("train_time_seconds", train_time)
    
    test_results = trainer_prompt.evaluate(tokenized_datasets['test'])
    mlflow.log_metric("test_accuracy", test_results['eval_accuracy'])
    mlflow.log_metric("test_f1_macro", test_results['eval_f1_macro'])
    print("Test Results:", test_results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 8,451 || all params: 109,492,998 || trainable%: 0.0077


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.239826,0.334601,0.214683
2,No log,1.135812,0.361217,0.232304
3,No log,1.101403,0.292776,0.262005


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
No log,1.106893,3,0.305660,0.279658


Test Results: {'eval_loss': 1.1068934202194214, 'eval_accuracy': 0.30566037735849055, 'eval_f1_macro': 0.2796578412028327}
🏃 View run Prompt_Tuning at: http://13.235.68.121:5000/#/experiments/1/runs/d46376dd07c74c618e071dafc5eb3550
🧪 View experiment at: http://13.235.68.121:5000/#/experiments/1
